# Chain in LangChain

## Overview

**Chain** is a core concept in LangChain that allows combining multiple components (LLM, prompts, parsers, tools, etc.) into a complete workflow.

## Chain Type Diagrams

### 1. Simple Chain

```
┌────────┐     ┌──────────────┐     ┌─────┐     ┌──────────────┐     ┌────────┐
│  Input │ --> │    Prompt    │ --> │ LLM │ --> │    Parser    │ --> │ Output │
│        │     │   Template   │     │     │     │              │     │        │
└────────┘     └──────────────┘     └─────┘     └──────────────┘     └────────┘
```

**Flow**: Input → Prompt Template → LLM → Output Parser → Output

---

### 2. Sequential Chain

```
┌─────────┐     ┌──────────┐     ┌──────────┐     ┌──────────┐     ┌────────┐
│  Input  │ --> │  Chain 1 │ --> │  Chain 2 │ --> │  Chain 3 │ --> │ Output │
│         │     │          │     │          │     │          │     │        │
└─────────┘     └──────────┘     └──────────┘     └──────────┘     └────────┘
                    │                │                │
                    ▼                ▼                ▼
                Output 1         Output 2         Output 3
                (Input 2)        (Input 3)        (Final)
```

**Flow**: Each chain's output becomes the next chain's input

---

### 3. Parallel Chain

```
                    ┌──────────┐
                    │  Chain 1 │──┐
                    └──────────┘  │
                                  │
┌─────────┐     ┌──────────┐      ├──> ┌──────────┐     ┌────────┐
│  Input  │ --> │  Parallel│ -->  │    │ Combine  │ --> │ Output │
│         │     │  Process │      ├──> │ Results  │     │        │
└─────────┘     └──────────┘      │    └──────────┘     └────────┘
                    │             │
                    ▼             │
                ┌──────────┐      │
                │  Chain 2 │ ─────┘
                └──────────┘
                  
```

**Flow**: Multiple chains execute simultaneously, then results are combined

---

### 4. Conditional Chain

```
┌─────────┐     ┌──────────────┐
│  Input  │ --> │   Condition  │
│         │     │    Check     │
└─────────┘     └──────────────┘
                       │
        ┌──────────────┼──────────────┐
        │              │              │
        ▼              ▼              ▼
   ┌─────────┐   ┌─────────┐   ┌─────────┐
   │ Branch 1│   │ Branch 2│   │ Branch 3│
   │ (True)  │   │ (False) │   │ (Else)  │
   └─────────┘   └─────────┘   └─────────┘
        │              │              │
        └──────────────┼──────────────┘
                       │
                       ▼
                 ┌────────┐
                 │ Output │
                 └────────┘
```

**Flow**: Input → Condition Check → Route to appropriate branch → Output

---

### 5. Complex Chain (Combined)

```
┌─────────┐
│  Input  │
└────┬────┘
     │
     ├─────────────────┐
     │                 │
     ▼                 ▼
┌──────────┐     ┌──────────┐
│ Parallel │     │ Parallel │
│  Chain 1 │     │  Chain 2 │
└────┬─────┘     └────┬─────┘
     │                │
     └────────┬───────┘
              │
              ▼
     ┌──────────────┐
     │   Sequential │
     │    Chain 3   │
     └──────┬───────┘
            │
            ▼
     ┌──────────────┐
     │  Conditional │
     │    Routing   │
     └──────┬───────┘
            │
            ▼
      ┌────────┐
      │ Output │
      └────────┘
```

**Flow**: Combines parallel, sequential, and conditional processing

## 1. Simple Chain 
 **Prompt → LLM → Output Parser**

```
Input → Prompt Template → LLM → Output Parser → Output
```

### Ví dụ:

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from config import settings

# Initialize LLM
llm = ChatOpenAI(
    model=settings.LLM_CHAT_MODEL,
    api_key=settings.LLM_API_KEY,
    base_url=settings.LLM_BASE_URL,
    temperature=0.7
)

# Create prompt template
prompt = ChatPromptTemplate.from_template(
    "Hãy giới thiệu về {topic} bằng tiếng Việt trong 3 câu."
)

# Create simple chain: Prompt → LLM → Parser
simple_chain = prompt | llm | StrOutputParser()

# Use chain
result = simple_chain.invoke({"topic": "Trí tuệ nhân tạo"})
print(result)

Dưới đây là giới thiệu về Trí tuệ nhân tạo trong 3 câu:

1. Trí tuệ nhân tạo (AI) là một lĩnh vực của khoa học máy tính tập trung vào việc tạo ra các hệ thống máy móc có khả năng mô phỏng trí thông minh và hành vi của con người.
2. Công nghệ này cho phép máy tính học hỏi từ dữ liệu, xử lý thông tin phức tạp và thực hiện các nhiệm vụ như giải quyết vấn đề hay ra quyết định một cách tự động.
3. Hiện nay, AI đang được ứng dụng rộng rãi trong nhiều lĩnh vực từ y tế, giáo dục đến giao thông, góp phần thay đổi diện mạo và nâng cao chất lượng cuộc sống hiện đại.


## 2. Sequential Chain - Chain tuần tự

Sequential chain cho phép kết nối nhiều bước, output của bước trước là input của bước sau.

```
Input → Chain 1 → Chain 2 → Chain 3 → Output
```

### Ví dụ: Chain 2 bước
1. Bước 1: Tạo câu hỏi về một chủ đề
2. Bước 2: Trả lời câu hỏi đó

In [2]:
# Chain 1: Generate question
question_prompt = ChatPromptTemplate.from_template(
    "Hãy tạo một câu hỏi thú vị về {topic}"
)
question_chain = question_prompt | llm | StrOutputParser()

# Chain 2: Answer the question
answer_prompt = ChatPromptTemplate.from_template(
    "Câu hỏi: {question}\n\nHãy trả lời câu hỏi này một cách chi tiết."
)

# Sequential chain: Chain 1 → Chain 2
sequential_chain = (
    {"question": question_chain}
    | answer_prompt
    | llm
    | StrOutputParser()
)

result = sequential_chain.invoke({"topic": "Machine Learning"})
print(result)

Câu hỏi về **"Bài toán xe tự lái và Đạo đức AI"** (một biến thể của *Trolley Problem - Bài toán xe điện*) là một trong những chủ đề gây tranh cãi nhất trong giới công nghệ, luật pháp và triết học hiện nay.

Dưới đây là phân tích chi tiết dựa trên các góc độ khác nhau để trả lời cho câu hỏi: **AI nên chọn thế nào và ai chịu trách nhiệm?**

---

### 1. AI nên chọn thế nào? (Góc độ Triết học và Thuật toán)

Khi lập trình cho tình huống này, các nhà phát triển thường phải đối mặt với hai trường phái triết học chính:

*   **Thuyết Vị lợi (Utilitarianism):** Ưu tiên cứu số đông. Theo logic này, AI sẽ chọn lao xuống vực để cứu 5 người. Mục tiêu là giảm thiểu tối đa thiệt hại về mạng người.
*   **Thuyết Nghĩa vụ (Deontology) & Quyền lợi cá nhân:** Một chiếc xe được thiết kế để bảo vệ hành khách bên trong. Nếu người tiêu dùng biết rằng chiếc xe họ mua có thể "phản bội" và chọn hy sinh họ trong một tình huống ngặt nghèo, sẽ không ai muốn mua xe tự lái nữa. Nếu không ai mua xe tự lái (vốn an toàn

## 3. Parallel Chain - Chain xử lý song song

Parallel chain cho phép xử lý nhiều tasks cùng lúc, sau đó kết hợp kết quả.

```
Input → [Chain 1, Chain 2, Chain 3] → Combine → Output
```

### Ưu điểm:
- **Tăng tốc độ**: Xử lý song song thay vì tuần tự
- **Hiệu quả**: Tận dụng tối đa tài nguyên
- **Linh hoạt**: Có thể xử lý các tasks độc lập cùng lúc

In [4]:
from langchain_core.runnables import RunnableParallel

# Create separate chains
summary_prompt = ChatPromptTemplate.from_template(
    "Tóm tắt ngắn gọn về {topic} trong 2 câu."
)
summary_chain = summary_prompt | llm | StrOutputParser()

pros_prompt = ChatPromptTemplate.from_template(
    "Liệt kê 3 ưu điểm của {topic}."
)
pros_chain = pros_prompt | llm | StrOutputParser()

cons_prompt = ChatPromptTemplate.from_template(
    "Liệt kê 3 nhược điểm của {topic}."
)
cons_chain = cons_prompt | llm | StrOutputParser()

# Parallel chain: Execute 3 chains simultaneously
parallel_chain = RunnableParallel({
    "summary": summary_chain,
    "pros": pros_chain,
    "cons": cons_chain
})

# Use chain
result = parallel_chain.invoke({"topic": "Trí tuệ nhân tạo"})
print(" Kết quả xử lý song song:")
print(f"\n Tóm tắt: {result['summary']}")
print(f"\n Ưu điểm: {result['pros']}")
print(f"\n Nhược điểm: {result['cons']}")

 Kết quả xử lý song song:

 Tóm tắt: Dưới đây là tóm tắt về Trí tuệ nhân tạo (AI) trong 2 câu:

Trí tuệ nhân tạo (AI) là lĩnh vực khoa học máy tính tập trung vào việc tạo ra các hệ thống có khả năng mô phỏng các quá trình trí tuệ của con người như học tập, suy luận và giải quyết vấn đề. Thông qua việc phân tích dữ liệu lớn, AI có thể tự động hóa các tác vụ phức tạp, đưa ra dự đoán và tương tác thông minh để hỗ trợ con người trong nhiều lĩnh vực đời sống.

 Ưu điểm: Dưới đây là 3 ưu điểm nổi bật của Trí tuệ nhân tạo (AI):

1.  **Tự động hóa và tăng năng suất:** AI có khả năng thực hiện các công việc lặp đi lặp lại một cách nhanh chóng và chính xác hơn con người. Điều này giúp giải phóng sức lao động, giảm thiểu sai sót do yếu tố con người và cho phép chúng ta tập trung vào những công việc sáng tạo, phức tạp hơn.
2.  **Xử lý và phân tích dữ liệu khổng lồ:** AI có thể phân tích hàng tỷ dữ liệu trong thời gian cực ngắn để tìm ra các quy luật, xu hướng mà mắt thường không thể nhận ra. Khả n

## 4. Conditional Chain - Chain có điều kiện

Conditional chain cho phép rẽ nhánh dựa trên điều kiện, tạo logic phức tạp hơn.

```
Input → Condition Check → [Branch 1 | Branch 2 | Branch 3] → Output
```

### Use cases:
- Routing dựa trên nội dung input
- Xử lý khác nhau cho các trường hợp khác nhau
- Error handling và fallback

In [5]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

# Chain for technical topics
technical_prompt = ChatPromptTemplate.from_template(
    "Giải thích {topic} một cách kỹ thuật và chi tiết."
)
technical_chain = technical_prompt | llm | StrOutputParser()

# Chain for general topics
general_prompt = ChatPromptTemplate.from_template(
    "Giải thích {topic} một cách đơn giản, dễ hiểu cho người mới bắt đầu."
)
general_chain = general_prompt | llm | StrOutputParser()

# Condition check function
def is_technical(topic: str) -> bool:
    technical_keywords = ["python", "machine learning", "ai", "algorithm", "code", "programming"]
    return any(keyword in topic.lower() for keyword in technical_keywords)

# Conditional chain
conditional_chain = RunnableBranch(
    # If technical topic → use technical chain
    (lambda x: is_technical(x.get("topic", "")), technical_chain),
    # Otherwise → use general chain
    general_chain
)

# Test with different topics
print("Technical topic:")
result1 = conditional_chain.invoke({"topic": "Python programming"})
print(result1[:200] + "...\\n")

print(" General topic:")
result2 = conditional_chain.invoke({"topic": "Lịch sử Việt Nam"})
print(result2[:200] + "...")

Technical topic:
Để giải thích về Python một cách kỹ thuật và chi tiết, chúng ta cần đi sâu vào kiến trúc hệ thống, cơ chế thực thi, quản lý bộ nhớ và các triết lý thiết kế đặc trưng của nó.

Dưới đây là phân tích chi...\n
 General topic:
Lịch sử Việt Nam giống như một bộ phim hành động dài tập, đầy kịch tính với tinh thần xuyên suốt là: **"Dựng nước và Giữ nước"**. Để dễ hiểu nhất, chúng ta có thể chia lịch sử Việt Nam thành 5 giai đo...


## 5. Complex Chain - Chain phức tạp kết hợp nhiều tính năng

Chain phức tạp kết hợp:
- Sequential processing
- Parallel processing  
- Conditional routing
- Custom transformations
- Error handling

### Ví dụ: Chain phân tích công ty
1. Trích xuất thông tin công ty (parallel với keywords)
2. Phân tích điểm mạnh/yếu
3. Đưa ra khuyến nghị
4. Route dựa trên điểm số

In [ ]:
from langchain_core.runnables import RunnableMap, RunnableLambda
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

# Pydantic model for structured output
class CompanyAnalysis(BaseModel):
    name: str = Field(description="Tên công ty")
    summary: str = Field(description="Tóm tắt")
    score: int = Field(description="Điểm từ 0-100", ge=0, le=100)

# Parser
parser = PydanticOutputParser(pydantic_object=CompanyAnalysis)

# Step 1: Extract company info
info_prompt = ChatPromptTemplate.from_template(
    "Cung cấp thông tin về công ty {company_name}.\\n{format_instructions}"
).partial(format_instructions=parser.get_format_instructions())

info_chain = info_prompt | llm | parser

# Step 2: Analyze company
analysis_prompt = ChatPromptTemplate.from_template(
    "Phân tích công ty {company_name} và đưa ra điểm số từ 0-100."
)
analysis_chain = analysis_prompt | llm | StrOutputParser()

# Step 3: Combine results
def combine_data(data):
    return {
        "company_info": data.get("company_info", {}),
        "analysis": data.get("analysis", "")
    }

# Complex chain combining parallel and sequential processing
complex_chain = (
    RunnableMap({
        "company_info": info_chain,
        "analysis": analysis_chain
    })
    | RunnableLambda(combine_data)
)

# Use chain
result = complex_chain.invoke({"company_name": "Rikkeisoft"})
print(result)